# Food inflation in Turkey

Can exchange rates and global food prices help predict monthly food inflation in Turkey?

- Monthly Turkish food CPI and four food categories: [TÜİK](https://veriportali.tuik.gov.tr/tr/press/58289), via [TCMB EVDS](https://evds3.tcmb.gov.tr/).
- Daily USD/TRY buying rates: [TCMB EVDS](https://evds3.tcmb.gov.tr/).
- Monthly global food-price indices: [FAO](https://www.fao.org/worldfoodsituation/foodpricesindex/en/).

## Setup

Use the project's uv environment as the notebook kernel. From the repository root, `uv run food-inflation-download` makes the source CSVs available in `data/raw/`. The command reuses any existing CSVs. An EVDS key is only needed to fetch missing EVDS files.

EVDS data starts in January 2005. [EVDS dates](https://www.tcmb.gov.tr/wps/wcm/connect/bab69efb-d66c-45b5-91c6-1533886acd6e/GenelAg-SSS.pdf?CACHE=NONE&CONTENTCACHE=NONE&MOD=AJPERES&attachment=true) refer to rates determined on the previous business day.

In [1]:
from pathlib import Path

import pandas as pd

from food_inflation.data_preparation import load_sources, monthly_indicators
from food_inflation.ingestion.evds import FOOD_CPI_FILE, USD_TRY_FILE
from food_inflation.ingestion.fao import FAO_FILE

project_root = Path.cwd().resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent
raw_dir = project_root / "data" / "raw"

## Load the source tables

Convert dates and indicator columns to their working types. Preparation removes EVDS timestamp metadata and requires each source to have unique dates in order.

In [2]:
cpi, usd_try, fao = load_sources(raw_dir)

## Proposal data check

### Source file sizes

Column counts include dates and metadata.

In [3]:
source_files = (FOOD_CPI_FILE, USD_TRY_FILE, FAO_FILE)
source_sizes = pd.DataFrame({
    "Rows": [len(cpi), len(usd_try), len(fao)],
    "Columns": [len(pd.read_csv(raw_dir / name, nrows=0).columns) for name in source_files],
    "One row": ["One month", "One calendar day", "One month"],
}, index=["Turkish CPI", "USD/TRY", "FAO"])
source_sizes

,Rows,Columns,One row
Turkish CPI,261,7,One month
USD/TRY,7943,3,One calendar day
FAO,441,7,One month


The following checks use the typed tables.

### Date coverage

In [4]:
source_coverage = pd.DataFrame({
    "First date": [cpi.index.min().date(), usd_try.index.min().date(), fao.index.min().date()],
    "Last date": [cpi.index.max().date(), usd_try.index.max().date(), fao.index.max().date()],
    "Numeric series": [cpi.shape[1], usd_try.shape[1], fao.shape[1]],
}, index=["Turkish CPI", "USD/TRY", "FAO"])
source_coverage

,First date,Last date,Numeric series
Turkish CPI,2005-01-01,2026-09-01,5
USD/TRY,2005-01-01,2026-09-30,1
FAO,1990-01-01,2026-09-01,6


### Date checks

In [5]:
date_checks = pd.DataFrame({
    "All dates present": [cpi.index.notna().all(), usd_try.index.notna().all(), fao.index.notna().all()],
    "Unique dates": [cpi.index.is_unique, usd_try.index.is_unique, fao.index.is_unique],
    "In date order": [cpi.index.is_monotonic_increasing, usd_try.index.is_monotonic_increasing, fao.index.is_monotonic_increasing],
    "Missing calendar rows": [
        len(pd.date_range(cpi.index.min(), cpi.index.max(), freq="MS").difference(cpi.index)),
        len(pd.date_range(usd_try.index.min(), usd_try.index.max(), freq="D").difference(usd_try.index)),
        len(pd.date_range(fao.index.min(), fao.index.max(), freq="MS").difference(fao.index)),
    ],
}, index=["Turkish CPI", "USD/TRY", "FAO"])
assert date_checks[["All dates present", "Unique dates", "In date order"]].all().all()
date_checks

,All dates present,Unique dates,In date order,Missing calendar rows
Turkish CPI,True,True,True,0
USD/TRY,True,True,True,0
FAO,True,True,True,0


### Missing numeric values

In [6]:
source_missing = pd.concat({
    "Turkish CPI": cpi.isna().sum(),
    "USD/TRY": usd_try.isna().sum(),
    "FAO": fao.isna().sum(),
}, names=["Source", "Column"]).to_frame("Missing values")
source_missing

Missing values
Source      Column                          
Turkish CPI food_cpi                       0
            cereals_cpi                    0
            meat_cpi                       0
            dairy_eggs_cpi                 0
            oils_fats_cpi                  0
USD/TRY     usd_try                     2470
FAO         food_price_index               0
            meat                           0
            dairy                          0
            cereals                        0
            vegetable_oils                 0
            sugar                          0

### Missing daily exchange rates

In [7]:
missing_rate = usd_try["usd_try"].isna()
weekend = usd_try.index.dayofweek >= 5

exchange_rate_counts = pd.Series({
    "Published rates": int((~missing_rate).sum()),
    "Missing rates": int(missing_rate.sum()),
    "Missing on weekends": int((missing_rate & weekend).sum()),
    "Missing on weekdays": int((missing_rate & ~weekend).sum()),
}, name="Daily observations").to_frame()
exchange_rate_counts

/var/folders/y6/fm262sgd3yz5p9hrjtwdtf5r0000gn/T/ipykernel_51289/248012972.py:2: Pandas4Warning: DatetimeIndex.dayofweek is deprecated and will be removed in a future version. Use DatetimeIndex.day_of_week instead.
  weekend = usd_try.index.dayofweek >= 5


,Daily observations
Published rates,5473
Missing rates,2470
Missing on weekends,2270
Missing on weekdays,200


## Monthly table

Average available daily rates within each month and keep every month in the CPI date range. `uv run food-inflation-prepare` applies the same preparation and saves `data/interim/monthly_indicators.parquet`.

In [8]:
monthly = monthly_indicators(cpi, usd_try, fao)
monthly.head()

,food_cpi,cereals_cpi,meat_cpi,dairy_eggs_cpi,oils_fats_cpi,usd_try,fao_food_price_index,fao_meat,fao_dairy,fao_cereals,fao_vegetable_oils,fao_sugar
date,,,,,,,,,,,,
2005-01-01,2.50,2.48,2.01,3.07,2.88,1.350042,65.8,69.9,76.2,60.5,63.15,54.0
2005-02-01,2.52,2.49,2.04,3.08,2.89,1.310155,66.2,71.1,76.7,60.3,62.27,56.3
2005-03-01,2.52,2.49,2.06,3.07,2.89,1.304957,67.5,71.7,77.0,61.5,66.74,55.0
2005-04-01,2.50,2.49,2.15,3.03,2.87,1.353514,66.0,70.4,76.2,59.5,65.07,53.2
2005-05-01,2.50,2.49,2.16,2.96,2.86,1.365048,66.5,72.6,76.5,59.4,64.23,53.1


### Monthly summary

The table contains index levels and exchange rates.

In [9]:
monthly_summary = pd.Series({
    "Monthly rows": len(monthly),
    "Numeric columns": monthly.shape[1],
    "First month": monthly.index.min().strftime("%B %Y"),
    "Last month": monthly.index.max().strftime("%B %Y"),
    "Missing numeric values": int(monthly.isna().sum().sum()),
    "Months missing any value": int(monthly.isna().any(axis=1).sum()),
    "Months without an exchange rate": int(monthly["usd_try"].isna().sum()),
}, name="Monthly coverage").to_frame()
monthly_summary

,Monthly coverage
Monthly rows,261
Numeric columns,12
First month,January 2005
Last month,September 2026
Missing numeric values,0
Months missing any value,0
Months without an exchange rate,0


In [10]:
monthly.isna().sum().to_frame("Missing monthly values")

,Missing monthly values
food_cpi,0
cereals_cpi,0
meat_cpi,0
dairy_eggs_cpi,0
oils_fats_cpi,0
usd_try,0
fao_food_price_index,0
fao_meat,0
fao_dairy,0
fao_cereals,0
